# Análisis Exploratorio (EDA) — Superstore: pandas vs PySpark vs Spark SQL

**Autor:** Eduardo Osorio Venegas
**Dataset:** `SuperStore_Tablon.xlsx` (9.800 pedidos, 18 columnas, 2015–2018)

Mismo análisis, **tres sintaxis distintas**, para que se vea dónde cambian de verdad y dónde son casi calcadas.

| | pandas | PySpark (DataFrame API) | Spark SQL |
|---|---|---|---|
| Motor de ejecución | En memoria, un solo proceso | Distribuido (Catalyst + Spark engine) | **El mismo motor que PySpark** |
| Evaluación | Eager | Lazy (hasta una acción) | Lazy (hasta una acción) |
| Sintaxis | Métodos encadenados de Python | Métodos encadenados de Python | Texto SQL declarativo |
| ¿Necesita índice? | Sí, tiene índice | No | No |
| Rendimiento | — | Igual que Spark SQL (mismo plan optimizado) | Igual que PySpark DataFrame API |

**El punto clave de esta comparación:** PySpark DataFrame API y Spark SQL **no son motores distintos** — el DataFrame API se traduce internamente al mismo plan lógico que genera una consulta SQL, y ambos pasan por el mismo optimizador (Catalyst). La elección entre uno y otro es **de estilo/legibilidad del equipo**, no de performance.


## 0. Setup — la misma tabla, tres formas de consultarla

Para poder usar SQL sobre un DataFrame de Spark, hace falta registrarlo como una **vista temporal** con `createOrReplaceTempView()` — a partir de ahí, `spark.sql("...")` lo consulta como si fuera una tabla.


In [1]:
import pandas as pd
from pyspark.sql import functions as F

ruta_archivo = "/lakehouse/default/Files/EDUARDO OSORIO/XLSX/SuperStore_Tablon.xlsx"

# --- pandas: DataFrame en memoria, con índice ---
pdf = pd.read_excel(ruta_archivo)
pdf["Order_Date"] = pd.to_datetime(pdf["Order_Date"])
pdf["Ship_Date"] = pd.to_datetime(pdf["Ship_Date"])

# --- PySpark: mismo contenido, como DataFrame distribuido ---
df = spark.createDataFrame(pdf)

# --- Vista temporal para poder usar Spark SQL sobre el mismo DataFrame ---
df.createOrReplaceTempView("superstore")

print(f"pandas    -> {type(pdf)}")
print(f"PySpark   -> {type(df)}")
print("Spark SQL -> vista temporal 'superstore' lista para consultar")

StatementMeta(, bb8974aa-2937-4ed7-aa9c-755ddf7d8ac2, 3, Finished, Available, Finished, False)

pandas    -> <class 'pandas.core.frame.DataFrame'>
PySpark   -> <class 'pyspark.sql.dataframe.DataFrame'>
Spark SQL -> vista temporal 'superstore' lista para consultar


## 1. Dimensiones del dataset

En SQL no existe un "shape": el conteo de filas es una consulta de agregación (`COUNT(*)`), y la cantidad de columnas se obtiene con `DESCRIBE`, no con la misma consulta.


In [2]:
# pandas — eager, inmediato
n_filas, n_cols = pdf.shape
print(f"pandas: {n_filas} filas x {n_cols} columnas")

StatementMeta(, bb8974aa-2937-4ed7-aa9c-755ddf7d8ac2, 4, Finished, Available, Finished, False)

pandas: 9800 filas x 18 columnas


In [3]:
# PySpark — count() es una acción
n_filas = df.count()
n_cols = len(df.columns)
print(f"PySpark: {n_filas} filas x {n_cols} columnas")


StatementMeta(, bb8974aa-2937-4ed7-aa9c-755ddf7d8ac2, 5, Finished, Available, Finished, False)

PySpark: 9800 filas x 18 columnas


In [4]:
%%sql
SELECT COUNT(*) AS total_filas
FROM superstore

StatementMeta(, bb8974aa-2937-4ed7-aa9c-755ddf7d8ac2, 6, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

Para ver la cantidad de columnas en SQL hace falta una consulta aparte: `DESCRIBE superstore` (ver sección 2) — no hay un equivalente directo a `.shape`.

## 2. Esquema y tipos de datos


In [6]:
# pandas
print(pdf.dtypes)

StatementMeta(, bb8974aa-2937-4ed7-aa9c-755ddf7d8ac2, 8, Finished, Available, Finished, False)

Row_ID                    int64
Order_ID                 object
Order_Date       datetime64[ns]
Ship_Date        datetime64[ns]
Ship_Mode                object
Customer_ID              object
Customer_Name            object
Segment                  object
Country                  object
City                     object
State                    object
Postal_Code             float64
Region                   object
Product_ID               object
Category                 object
Sub_Category             object
Product_Name             object
Sales                   float64
dtype: object


In [7]:
# PySpark
df.printSchema()

StatementMeta(, bb8974aa-2937-4ed7-aa9c-755ddf7d8ac2, 9, Finished, Available, Finished, False)

root
 |-- Row_ID: long (nullable = true)
 |-- Order_ID: string (nullable = true)
 |-- Order_Date: timestamp (nullable = true)
 |-- Ship_Date: timestamp (nullable = true)
 |-- Ship_Mode: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal_Code: double (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product_ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub_Category: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Sales: double (nullable = true)



In [8]:
%%sql
DESCRIBE superstore


StatementMeta(, bb8974aa-2937-4ed7-aa9c-755ddf7d8ac2, 10, Finished, Available, Finished, False)

<Spark SQL result set with 18 rows and 3 fields>

## 3. Primeras filas


In [9]:
# pandas
pdf.head(5)

StatementMeta(, bb8974aa-2937-4ed7-aa9c-755ddf7d8ac2, 13, Finished, Available, Finished, False)

,Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,Country,City,State,Postal_Code,Region,Product_ID,Category,Sub_Category,Product_Name,Sales
0,1,CA-2017-152156,2017-11-08,2017-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96
1,2,CA-2017-152156,2017-11-08,2017-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.94
2,3,CA-2017-138688,2017-06-12,2017-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036.0,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.62
3,4,US-2016-108966,2016-10-11,2016-10-18,Standard Class,SO-20335,Sean O Donnel,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,9575775.00
4,5,US-2016-108966,2016-10-11,2016-10-18,Standard Class,SO-20335,Sean O Donnel,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold N Roll Cart System,22368.00


In [10]:
# PySpark
df.show(5, truncate=False)

StatementMeta(, bb8974aa-2937-4ed7-aa9c-755ddf7d8ac2, 16, Finished, Available, Finished, False)

+------+--------------+-------------------+-------------------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+-----------------------------------------------------------+---------+
|Row_ID|Order_ID      |Order_Date         |Ship_Date          |Ship_Mode     |Customer_ID|Customer_Name  |Segment  |Country      |City           |State     |Postal_Code|Region|Product_ID     |Category       |Sub_Category|Product_Name                                               |Sales    |
+------+--------------+-------------------+-------------------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+-----------------------------------------------------------+---------+
|1     |CA-2017-152156|2017-11-08 00:00:00|2017-11-11 00:00:00|Second Class  |CG-12520   |Claire Gute    |Consumer |United S

In [11]:
display(df.limit(5))

StatementMeta(, bb8974aa-2937-4ed7-aa9c-755ddf7d8ac2, 17, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, cc6265f4-a8fa-436d-917d-7f3836266dc4)

In [12]:
%%sql
SELECT *
FROM superstore
LIMIT 5

StatementMeta(, bb8974aa-2937-4ed7-aa9c-755ddf7d8ac2, 18, Finished, Available, Finished, False)

<Spark SQL result set with 5 rows and 18 fields>

## 4. Valores nulos por columna

Acá SQL es el más verboso de los tres: no hay una función que revise "todas las columnas" a la vez — hay que escribir el `SUM(CASE WHEN ... END)` una vez por columna (igual que PySpark sin la comprensión de listas de Python).


In [13]:
# pandas
pdf.isnull().sum()

StatementMeta(, bb8974aa-2937-4ed7-aa9c-755ddf7d8ac2, 19, Finished, Available, Finished, False)

Row_ID            0
Order_ID          0
Order_Date        0
Ship_Date         0
Ship_Mode         0
Customer_ID       0
Customer_Name     0
Segment           0
Country           0
City              0
State             0
Postal_Code      11
Region            0
Product_ID        0
Category          0
Sub_Category      0
Product_Name      0
Sales             0
dtype: int64

In [14]:
# PySpark
df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show(vertical=True)


StatementMeta(, bb8974aa-2937-4ed7-aa9c-755ddf7d8ac2, 20, Finished, Available, Finished, False)

-RECORD 0------------
 Row_ID        | 0   
 Order_ID      | 0   
 Order_Date    | 0   
 Ship_Date     | 0   
 Ship_Mode     | 0   
 Customer_ID   | 0   
 Customer_Name | 0   
 Segment       | 0   
 Country       | 0   
 City          | 0   
 State         | 0   
 Postal_Code   | 11  
 Region        | 0   
 Product_ID    | 0   
 Category      | 0   
 Sub_Category  | 0   
 Product_Name  | 0   
 Sales         | 0   



In [15]:
%%sql
SELECT
    SUM(CASE WHEN Row_ID        IS NULL THEN 1 ELSE 0 END) AS Row_ID,
    SUM(CASE WHEN Order_ID      IS NULL THEN 1 ELSE 0 END) AS Order_ID,
    SUM(CASE WHEN Order_Date    IS NULL THEN 1 ELSE 0 END) AS Order_Date,
    SUM(CASE WHEN Postal_Code   IS NULL THEN 1 ELSE 0 END) AS Postal_Code,
    SUM(CASE WHEN Region        IS NULL THEN 1 ELSE 0 END) AS Region,
    SUM(CASE WHEN Sales         IS NULL THEN 1 ELSE 0 END) AS Sales
    -- (se repite el mismo patrón para el resto de las 18 columnas)
FROM superstore

StatementMeta(, bb8974aa-2937-4ed7-aa9c-755ddf7d8ac2, 21, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 6 fields>

En pandas y PySpark, el bucle sobre `df.columns` evita escribir las 18 columnas a mano — en SQL puro no hay ese atajo, por eso en la práctica muchos equipos generan este SQL dinámicamente desde Python en vez de escribirlo a mano.


## 5. Cardinalidad de las columnas categóricas


In [16]:
# pandas
for col in ["Category", "Sub_Category", "Region", "Segment"]:
    print(f"{col}: {pdf[col].nunique()} valores únicos")

StatementMeta(, bb8974aa-2937-4ed7-aa9c-755ddf7d8ac2, 22, Finished, Available, Finished, False)

Category: 3 valores únicos
Sub_Category: 17 valores únicos
Region: 4 valores únicos
Segment: 3 valores únicos


In [17]:
# PySpark
df.select([
    F.countDistinct(c).alias(f"{c}_exact") for c in ["Category", "Sub_Category", "Region", "Segment"]
]).show()


StatementMeta(, bb8974aa-2937-4ed7-aa9c-755ddf7d8ac2, 23, Finished, Available, Finished, False)

+--------------+------------------+------------+-------------+
|Category_exact|Sub_Category_exact|Region_exact|Segment_exact|
+--------------+------------------+------------+-------------+
|             3|                17|           4|            3|
+--------------+------------------+------------+-------------+



In [18]:
%%sql
SELECT
    COUNT(DISTINCT Category)     AS Category_valores,
    COUNT(DISTINCT Sub_Category) AS Sub_Category_valores,
    COUNT(DISTINCT Region)       AS Region_valores,
    COUNT(DISTINCT Segment)      AS Segment_valores
FROM superstore

StatementMeta(, bb8974aa-2937-4ed7-aa9c-755ddf7d8ac2, 24, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 4 fields>

## 6. Frecuencia de categorías


In [19]:
# pandas
pdf["Category"].value_counts()

StatementMeta(, bb8974aa-2937-4ed7-aa9c-755ddf7d8ac2, 25, Finished, Available, Finished, False)

Category
Office Supplies    5909
Furniture          2078
Technology         1813
Name: count, dtype: int64

In [20]:
# PySpark
df.groupBy("Category").count().orderBy(F.desc("count")).show()

StatementMeta(, bb8974aa-2937-4ed7-aa9c-755ddf7d8ac2, 26, Finished, Available, Finished, False)

+---------------+-----+
|       Category|count|
+---------------+-----+
|Office Supplies| 5909|
|      Furniture| 2078|
|     Technology| 1813|
+---------------+-----+



In [22]:
%%sql
SELECT Category, COUNT(*) AS total
FROM superstore
GROUP BY Category
ORDER BY total DESC

StatementMeta(, bb8974aa-2937-4ed7-aa9c-755ddf7d8ac2, 28, Finished, Available, Finished, False)

<Spark SQL result set with 3 rows and 2 fields>

Acá **Spark SQL y PySpark DataFrame API son prácticamente un calco estructural**: `GROUP BY` + `ORDER BY` en SQL es exactamente `.groupBy()` + `.orderBy()` en el DataFrame API — no sorprende, porque el DataFrame API generó ese mismo plan por dentro.


## 7. Estadísticas descriptivas de `Sales`

Spark SQL no tiene un `describe()`/`summary()` de una palabra — hay que pedir cada estadístico como función de agregación, usando `percentile_approx()` para los cuartiles.


In [23]:
# pandas — incluye percentiles por defecto
pdf["Sales"].describe()

StatementMeta(, bb8974aa-2937-4ed7-aa9c-755ddf7d8ac2, 29, Finished, Available, Finished, False)

count    9.800000e+03
mean     1.014267e+05
std      5.215577e+05
min      4.440000e-01
25%      3.989500e+01
50%      3.563750e+02
75%      2.534400e+04
max      2.396266e+07
Name: Sales, dtype: float64

In [24]:
# PySpark
df.select("Sales").describe().show()
df.select("Sales").summary("count", "mean", "stddev", "min", "25%", "50%", "75%", "max").show()

StatementMeta(, bb8974aa-2937-4ed7-aa9c-755ddf7d8ac2, 30, Finished, Available, Finished, False)

+-------+------------------+
|summary|             Sales|
+-------+------------------+
|  count|              9800|
|   mean|101426.65048326526|
| stddev| 521557.7453120213|
|    min|             0.444|
|    max|       2.3962656E7|
+-------+------------------+

+-------+------------------+
|summary|             Sales|
+-------+------------------+
|  count|              9800|
|   mean|101426.65048326526|
| stddev| 521557.7453120213|
|    min|             0.444|
|    25%|             39.88|
|    50%|            355.36|
|    75%|           25344.0|
|    max|       2.3962656E7|
+-------+------------------+



In [25]:
%%sql
SELECT
    COUNT(Sales)                          AS count,
    AVG(Sales)                            AS mean,
    STDDEV(Sales)                         AS stddev,
    MIN(Sales)                            AS min,
    PERCENTILE_APPROX(Sales, 0.25)        AS p25,
    PERCENTILE_APPROX(Sales, 0.5)         AS p50,
    PERCENTILE_APPROX(Sales, 0.75)        AS p75,
    MAX(Sales)                            AS max
FROM superstore

StatementMeta(, bb8974aa-2937-4ed7-aa9c-755ddf7d8ac2, 31, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 8 fields>

## 8. Ventas totales por categoría


In [26]:
# pandas
pdf.groupby("Category")["Sales"].sum().sort_values(ascending=False)

StatementMeta(, bb8974aa-2937-4ed7-aa9c-755ddf7d8ac2, 32, Finished, Available, Finished, False)

Category
Furniture          5.115324e+08
Technology         2.757589e+08
Office Supplies    2.066898e+08
Name: Sales, dtype: float64

In [27]:
# PySpark
(
    df.groupBy("Category")
    .agg(F.sum("Sales").alias("Sales_total"))
    .orderBy(F.desc("Sales_total"))
    .show()
)

StatementMeta(, bb8974aa-2937-4ed7-aa9c-755ddf7d8ac2, 33, Finished, Available, Finished, False)

+---------------+--------------------+
|       Category|         Sales_total|
+---------------+--------------------+
|      Furniture|      5.1153240282E8|
|     Technology|       2.757589222E8|
|Office Supplies|2.0668984971600017E8|
+---------------+--------------------+



In [28]:
%%sql
SELECT Category, SUM(Sales) AS Sales_total
FROM superstore
GROUP BY Category
ORDER BY Sales_total DESC

StatementMeta(, bb8974aa-2937-4ed7-aa9c-755ddf7d8ac2, 34, Finished, Available, Finished, False)

<Spark SQL result set with 3 rows and 2 fields>

## 9. Top N por una columna


In [29]:
# pandas
pdf.nlargest(5, "Sales")[["Row_ID", "Product_Name", "Sales"]]

StatementMeta(, bb8974aa-2937-4ed7-aa9c-755ddf7d8ac2, 35, Finished, Available, Finished, False)

,Row_ID,Product_Name,Sales
399,400,"Riverside Palais Royal Lawyers Bookcase, Royal...",23962656.0
5055,5056,OSullivan Living Dimensions 5-Shelf Bookcases,13523976.0
8781,8782,"Atlantic Metals Mobile 5-Shelf Bookcases, Cust...",12279984.0
2623,2624,Canon imageCLASS 2200 Advanced Copier,11199968.0
3,4,Bretford CR4500 Series Slim Rectangular Table,9575775.0


In [30]:
# PySpark
(
    df.orderBy(F.desc("Sales"))
    .select("Row_ID", "Product_Name", "Sales")
    .limit(5)
    .show(truncate=False)
)

StatementMeta(, bb8974aa-2937-4ed7-aa9c-755ddf7d8ac2, 36, Finished, Available, Finished, False)

+------+-------------------------------------------------------------+-----------+
|Row_ID|Product_Name                                                 |Sales      |
+------+-------------------------------------------------------------+-----------+
|400   |Riverside Palais Royal Lawyers Bookcase, Royale Cherry Finish|2.3962656E7|
|5056  |OSullivan Living Dimensions 5-Shelf Bookcases                |1.3523976E7|
|8782  |Atlantic Metals Mobile 5-Shelf Bookcases, Custom Colors      |1.2279984E7|
|2624  |Canon imageCLASS 2200 Advanced Copier                        |1.1199968E7|
|4     |Bretford CR4500 Series Slim Rectangular Table                |9575775.0  |
+------+-------------------------------------------------------------+-----------+



In [31]:
%%sql
SELECT Row_ID, Product_Name, Sales
FROM superstore
ORDER BY Sales DESC
LIMIT 5

StatementMeta(, bb8974aa-2937-4ed7-aa9c-755ddf7d8ac2, 37, Finished, Available, Finished, False)

<Spark SQL result set with 5 rows and 3 fields>

## 10. Filtrado condicional


In [32]:
# pandas — boolean indexing
pdf[(pdf["Category"] == "Technology") & (pdf["Sales"] > 1000)].shape[0]

StatementMeta(, bb8974aa-2937-4ed7-aa9c-755ddf7d8ac2, 38, Finished, Available, Finished, False)

805

In [33]:
# PySpark
df.filter((F.col("Category") == "Technology") & (F.col("Sales") > 1000)).count()

StatementMeta(, bb8974aa-2937-4ed7-aa9c-755ddf7d8ac2, 39, Finished, Available, Finished, False)

805

In [34]:
%%sql
SELECT COUNT(*) AS total
FROM superstore
WHERE Category = 'Technology'
  AND Sales > 1000

StatementMeta(, bb8974aa-2937-4ed7-aa9c-755ddf7d8ac2, 40, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

## 11. Crear una columna derivada

En SQL no se "agrega" una columna a una tabla existente con un `SELECT` — se proyecta una columna calculada en el resultado (o se materializa con `CREATE TABLE AS` / `CREATE VIEW` si hace falta reutilizarla).


In [35]:
# pandas
pdf["Dias_envio"] = (pdf["Ship_Date"] - pdf["Order_Date"]).dt.days
pdf[["Order_Date", "Ship_Date", "Dias_envio"]].head()

StatementMeta(, bb8974aa-2937-4ed7-aa9c-755ddf7d8ac2, 41, Finished, Available, Finished, False)

,Order_Date,Ship_Date,Dias_envio
0,2017-11-08,2017-11-11,3
1,2017-11-08,2017-11-11,3
2,2017-06-12,2017-06-16,4
3,2016-10-11,2016-10-18,7
4,2016-10-11,2016-10-18,7


In [36]:
# PySpark
df = df.withColumn("Dias_envio", F.datediff(F.col("Ship_Date"), F.col("Order_Date")))
df.select("Order_Date", "Ship_Date", "Dias_envio").show(5)

StatementMeta(, bb8974aa-2937-4ed7-aa9c-755ddf7d8ac2, 42, Finished, Available, Finished, False)

+-------------------+-------------------+----------+
|         Order_Date|          Ship_Date|Dias_envio|
+-------------------+-------------------+----------+
|2017-11-08 00:00:00|2017-11-11 00:00:00|         3|
|2017-11-08 00:00:00|2017-11-11 00:00:00|         3|
|2017-06-12 00:00:00|2017-06-16 00:00:00|         4|
|2016-10-11 00:00:00|2016-10-18 00:00:00|         7|
|2016-10-11 00:00:00|2016-10-18 00:00:00|         7|
+-------------------+-------------------+----------+
only showing top 5 rows



In [37]:
%%sql
SELECT
    Order_Date,
    Ship_Date,
    DATEDIFF(Ship_Date, Order_Date) AS Dias_envio
FROM superstore
LIMIT 5

StatementMeta(, bb8974aa-2937-4ed7-aa9c-755ddf7d8ac2, 43, Finished, Available, Finished, False)

<Spark SQL result set with 5 rows and 3 fields>

## 12. Correlación entre columnas numéricas

Acá el resultado sorprende: para un **par** de columnas, `corr()` existe como función de agregación nativa en Spark SQL — y es más simple que el camino que hay que tomar en el DataFrame API (`VectorAssembler` + `pyspark.ml.stat.Correlation`), que está pensado para una **matriz completa** de muchas columnas a la vez, no para un par.


In [38]:
# pandas — matriz de correlación completa en una línea
pdf[["Sales", "Postal_Code"]].corr()

StatementMeta(, bb8974aa-2937-4ed7-aa9c-755ddf7d8ac2, 44, Finished, Available, Finished, False)

,Sales,Postal_Code
Sales,1.000000,0.031332
Postal_Code,0.031332,1.000000


In [39]:
# PySpark — atajo para un PAR de columnas (equivalente directo al SQL)
df.stat.corr("Sales", "Postal_Code")

# Para una MATRIZ completa (3+ columnas) sí hace falta el camino largo:
# from pyspark.ml.feature import VectorAssembler
# from pyspark.ml.stat import Correlation
# ensamblador = VectorAssembler(inputCols=[...], outputCol="features")
# Correlation.corr(ensamblador.transform(df), "features").head()[0]

StatementMeta(, bb8974aa-2937-4ed7-aa9c-755ddf7d8ac2, 45, Finished, Available, Finished, False)

0.03165183040867192

In [40]:
%%sql
SELECT corr(Sales, Postal_Code) AS correlacion
FROM superstore

StatementMeta(, bb8974aa-2937-4ed7-aa9c-755ddf7d8ac2, 46, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

En este dataset la correlación es prácticamente nula (~0.03) — esperable, porque `Postal_Code` es un código, no una magnitud continua.


## 13. Rango de fechas


In [41]:
# pandas
print(pdf["Order_Date"].min(), "->", pdf["Order_Date"].max())

StatementMeta(, bb8974aa-2937-4ed7-aa9c-755ddf7d8ac2, 47, Finished, Available, Finished, False)

2015-01-03 00:00:00 -> 2018-12-30 00:00:00


In [42]:
# PySpark
df.select(F.min("Order_Date"), F.max("Order_Date")).show()

StatementMeta(, bb8974aa-2937-4ed7-aa9c-755ddf7d8ac2, 48, Finished, Available, Finished, False)

+-------------------+-------------------+
|    min(Order_Date)|    max(Order_Date)|
+-------------------+-------------------+
|2015-01-03 00:00:00|2018-12-30 00:00:00|
+-------------------+-------------------+



In [43]:
%%sql
SELECT MIN(Order_Date) AS fecha_min, MAX(Order_Date) AS fecha_max
FROM superstore

StatementMeta(, bb8974aa-2937-4ed7-aa9c-755ddf7d8ac2, 49, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 2 fields>

## 14. Resumen: ¿cuál conviene para cada caso?

| Tarea | Mejor opción | Por qué |
|---|---|---|
| Dataset chico, exploración rápida | **pandas** | Sintaxis más corta, eager, sin overhead de un cluster |
| Dataset grande del Lakehouse/Warehouse | **PySpark o Spark SQL** (da igual cuál) | Mismo motor distribuido, misma performance |
| Analista/BI que ya piensa en SQL | **Spark SQL** | No hay que aprender la API de DataFrames |
| Ingeniero que encadena transformaciones complejas en código | **PySpark DataFrame API** | Se combina mejor con funciones Python, loops, listas de columnas |
| "Todas las columnas" con una comprensión de listas | **PySpark** | pandas y PySpark iteran `df.columns` en Python; SQL puro obliga a escribir cada columna a mano |
| Correlación de un par de columnas | **PySpark (`.stat.corr()`) o SQL (`corr()`)** | Ambos exponen el mismo atajo; pandas gana solo cuando se necesita la matriz completa |

**Conclusión:** la elección entre PySpark DataFrame API y Spark SQL no es una decisión de performance — es una decisión de **quién va a leer y mantener ese código**. Muchos notebooks de Fabric mezclan las dos libremente: DataFrame API para armar el pipeline de transformaciones, y `%%sql` / `spark.sql()` para la parte exploratoria o de validación final.
